# Extraction des masques

Notebook pour l'extraction et l'enregistrement de masques segmentés à partir d'images et d'un dataframe contenant les coordonnées des polygones. Les masques ont été obtenus grâce au fine-tuning de YOLOv11 réalisé sur la plateforme Arkindex.  
Léa Maronet - 2026

In [1]:
import pandas as pd
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ast
from shapely.geometry import Polygon, box
import os
from tqdm import tqdm

## Import des données

In [2]:
polygonesMasques = pd.read_csv("../data/polygonesMasques.csv", index_col=0)
corpusTropy = pd.read_csv("../data/corpusTropy.csv")
corpusTropy

,Titre,État,Région,Site,Description,Date,Support,Affiliation religieuse,Lieu de conservation,Auteur
0,ASOK_UTT059a,Uttar Pradesh,Uttar Pradesh,Kauśambī,NaN,1 A.D.,Portail ; architrave,bouddhisme,Allahabad Municipal Museum,Maronet Léa
1,ASOK_UTT031b,Uttar Pradesh,Mathurā,Vrindavan,Bodhisattva assis,1 A.D. ; 2 A.D.,Sculpture en ronde-bosse,bouddhisme,Mathura Government Museum,Maronet Léa
2,ASOK_UTT031a,Uttar Pradesh,Mathurā,Vrindavan,Bodhisattva assis,1 A.D. ; 2 A.D.,Sculpture en ronde-bosse,bouddhisme,Mathura Government Museum,Maronet Léa
3,ASOK_UTT026,Uttar Pradesh,Mathurā,Govindnāgar,Balustrade avec médaillons et demi-médaillons,1 A.D. ; 2 A.D.,Balustrade ; pilier,indéterminée,Mathura Government Museum,Maronet Léa
4,ASOK_UTT014,Uttar Pradesh,Mathurā,Govindnāgar,Balustrade avec médaillons et demi-médaillons,1 A.D. ; 2 A.D.,Balustrade ; pilier,indéterminée,Mathura Government Museum,Maronet Léa
...,...,...,...,...,...,...,...,...,...,...
134,ASOK_AMA007,Andhra Pradesh,Āndhradeśa,Amarāvatī,Pilier de balustrade avec médaillon lotiforme ...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; balustrade ; pilier,bouddhisme,The British Museum,Maronet Léa
135,ASOK_AMA005,Andhra Pradesh,Āndhradeśa,Amarāvatī,Pilier de balustrade avec médaillons lotiforme...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; balustrade ; pilier,bouddhisme,The British Museum,Maronet Léa
136,ASOK_AMA003,Andhra Pradesh,Āndhradeśa,Amarāvatī,Pilier de balustrade avec médaillons lotiforme...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; balustrade ; pilier,bouddhisme,The British Museum,Maronet Léa
137,ASOK_AMA002,Andhra Pradesh,Āndhradeśa,Amarāvatī,Pilier de balustrade avec médaillons lotiforme...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; balustrade ; pilier,bouddhisme,The British Museum,Maronet Léa


In [3]:
polygonesMasques

,Nom_masque,Motif,Polygone,Confiance,Nom_image
0,ASOK_AND003_lion_13.png,lion,"[[243, 2456], [335, 2548], [318, 2632], [402, ...",0.98,ASOK_AND003.JPG
1,ASOK_NAG002b_lotus_69.png,lotus,"[[182, 671], [184, 680], [1035, 680], [1035, 6...",0.99,ASOK_NAG002b.jpg
2,ASOK_NAG002b_lion_72.png,lion,"[[316, 802], [326, 806], [325, 815], [338, 841...",0.97,ASOK_NAG002b.jpg
3,ASOK_AND002_lotus_109.png,lotus,"[[1368, 708], [1382, 819], [1433, 897], [1490,...",0.99,ASOK_AND002.JPG
4,ASOK_AND002_lotus_110.png,lotus,"[[1358, 2966], [1376, 3139], [2016, 3148], [20...",0.99,ASOK_AND002.JPG
...,...,...,...,...,...
539,ASOK_AMA052_lotus_14203.png,lotus,"[[107, 2071], [139, 2233], [156, 2318], [168, ...",0.97,ASOK_AMA052.jpg
540,ASOK_AMA053_lotus_14208.png,lotus,"[[2, 2414], [2, 2565], [23, 2873], [581, 2862]...",0.98,ASOK_AMA053.jpg
541,ASOK_AMA072_lion_14211.png,lion,"[[681, 743], [694, 794], [710, 798], [731, 798...",1.00,ASOK_AMA072.jpg
542,ASOK_AMA072_lion_14212.png,lion,"[[87, 766], [88, 805], [109, 808], [130, 808],...",0.99,ASOK_AMA072.jpg


## Extraction et sauvegarde des masques

In [4]:
def extraction_polygones(df, marge_fixe=100, taille_sortie=512):
    df = df.reset_index(drop=True)
    os.makedirs("../data/masques", exist_ok=True)

    for i in tqdm(range(len(df))):
        image_nom = df.iloc[i]["Nom_image"]
        image_chemin = f"../data/images/{image_nom}"
        image = cv2.imread(image_chemin)
        masque_polygone = df.iloc[i]["Polygone"]
        polygone = np.array(ast.literal_eval(masque_polygone), dtype=np.int32)

        masque = np.zeros(image.shape[:2], dtype=np.uint8)
        cv2.fillPoly(masque, [polygone], color=255)
        masque_image = cv2.bitwise_and(image, image, mask=masque)

        x, y, w, h = cv2.boundingRect(polygone)
        x1 = max(x - marge_fixe, 0)
        y1 = max(y - marge_fixe, 0)
        x2 = min(x + w + marge_fixe, image.shape[1])
        y2 = min(y + h + marge_fixe, image.shape[0])

        cropped = masque_image[y1:y2, x1:x2]
        cropped_h, cropped_w = cropped.shape[:2]
        taille = max(cropped_h, cropped_w)
        square_image = np.zeros((taille, taille, 3), dtype=np.uint8)

        haut = (taille - cropped_h) // 2
        gauche = (taille - cropped_w) // 2
        square_image[haut:haut + cropped_h, gauche:gauche + cropped_w] = cropped

        image_finale = cv2.resize(square_image, (taille_sortie, taille_sortie), interpolation=cv2.INTER_AREA)
        masque_nom = df.iloc[i]["Nom_masque"]
        output_path = os.path.join("../data/masques", masque_nom)

        cv2.imwrite(output_path, image_finale)

    return df
    
extraction_polygones(polygonesMasques)

100%|██████████| 544/544 [01:02<00:00,  8.66it/s]


,Nom_masque,Motif,Polygone,Confiance,Nom_image
0,ASOK_AND003_lion_13.png,lion,"[[243, 2456], [335, 2548], [318, 2632], [402, ...",0.98,ASOK_AND003.JPG
1,ASOK_NAG002b_lotus_69.png,lotus,"[[182, 671], [184, 680], [1035, 680], [1035, 6...",0.99,ASOK_NAG002b.jpg
2,ASOK_NAG002b_lion_72.png,lion,"[[316, 802], [326, 806], [325, 815], [338, 841...",0.97,ASOK_NAG002b.jpg
3,ASOK_AND002_lotus_109.png,lotus,"[[1368, 708], [1382, 819], [1433, 897], [1490,...",0.99,ASOK_AND002.JPG
4,ASOK_AND002_lotus_110.png,lotus,"[[1358, 2966], [1376, 3139], [2016, 3148], [20...",0.99,ASOK_AND002.JPG
...,...,...,...,...,...
539,ASOK_AMA052_lotus_14203.png,lotus,"[[107, 2071], [139, 2233], [156, 2318], [168, ...",0.97,ASOK_AMA052.jpg
540,ASOK_AMA053_lotus_14208.png,lotus,"[[2, 2414], [2, 2565], [23, 2873], [581, 2862]...",0.98,ASOK_AMA053.jpg
541,ASOK_AMA072_lion_14211.png,lion,"[[681, 743], [694, 794], [710, 798], [731, 798...",1.00,ASOK_AMA072.jpg
542,ASOK_AMA072_lion_14212.png,lion,"[[87, 766], [88, 805], [109, 808], [130, 808],...",0.99,ASOK_AMA072.jpg


## Merge et export du dataframe

In [5]:
polygonesMasques["Titre"] = polygonesMasques["Nom_image"].apply(lambda x: os.path.splitext(x)[0])
corpusMasques = polygonesMasques.merge(corpusTropy, left_on="Titre", right_on="Titre", how="left")
corpusMasques["Polygone"] = corpusMasques["Polygone"].apply(ast.literal_eval).drop(columns=["Titre"])
corpusMasques.to_csv('../outputs/corpusMasques.csv', index=False)
corpusMasques

,Nom_masque,Motif,Polygone,Confiance,Nom_image,Titre,État,Région,Site,Description,Date,Support,Affiliation religieuse,Lieu de conservation,Auteur
0,ASOK_AND003_lion_13.png,lion,"[[243, 2456], [335, 2548], [318, 2632], [402, ...",0.98,ASOK_AND003.JPG,ASOK_AND003,Andhra Pradesh,Āndhradeśa,Dupāḍu,Animaux ailés,1 A.D.,Stūpa ; frise,bouddhisme,Amaravati Heritage Centre and Museum,Maronet Léa
1,ASOK_NAG002b_lotus_69.png,lotus,"[[182, 671], [184, 680], [1035, 680], [1035, 6...",0.99,ASOK_NAG002b.jpg,ASOK_NAG002b,Andhra Pradesh,Āndhradeśa,Nāgārjunakoṇḍa,Demi-médaillon lotiforme et lion,3 A.D.,Pilier,bouddhisme,In situ,Maronet Léa
2,ASOK_NAG002b_lion_72.png,lion,"[[316, 802], [326, 806], [325, 815], [338, 841...",0.97,ASOK_NAG002b.jpg,ASOK_NAG002b,Andhra Pradesh,Āndhradeśa,Nāgārjunakoṇḍa,Demi-médaillon lotiforme et lion,3 A.D.,Pilier,bouddhisme,In situ,Maronet Léa
3,ASOK_AND002_lotus_109.png,lotus,"[[1368, 708], [1382, 819], [1433, 897], [1490,...",0.99,ASOK_AND002.JPG,ASOK_AND002,Andhra Pradesh,Āndhradeśa,Dupāḍu,Portail avec cheval et médaillons lotiformes,1 A.D.,Stūpa ; frise,bouddhisme,Amaravati Heritage Centre and Museum,Maronet Léa
4,ASOK_AND002_lotus_110.png,lotus,"[[1358, 2966], [1376, 3139], [2016, 3148], [20...",0.99,ASOK_AND002.JPG,ASOK_AND002,Andhra Pradesh,Āndhradeśa,Dupāḍu,Portail avec cheval et médaillons lotiformes,1 A.D.,Stūpa ; frise,bouddhisme,Amaravati Heritage Centre and Museum,Maronet Léa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,ASOK_AMA052_lotus_14203.png,lotus,"[[107, 2071], [139, 2233], [156, 2318], [168, ...",0.97,ASOK_AMA052.jpg,ASOK_AMA052,Andhra Pradesh,Āndhradeśa,Amarāvatī,Pilier de balustrade avec médaillon et demi-mé...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; balustrade ; pilier,bouddhisme,Chennai Government Museum,Maronet Léa
540,ASOK_AMA053_lotus_14208.png,lotus,"[[2, 2414], [2, 2565], [23, 2873], [581, 2862]...",0.98,ASOK_AMA053.jpg,ASOK_AMA053,Andhra Pradesh,Āndhradeśa,Amarāvatī,Pilier de balustrade avec demi-médaillon lotif...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; balustrade ; pilier,bouddhisme,Chennai Government Museum,Maronet Léa
541,ASOK_AMA072_lion_14211.png,lion,"[[681, 743], [694, 794], [710, 798], [731, 798...",1.00,ASOK_AMA072.jpg,ASOK_AMA072,Andhra Pradesh,Āndhradeśa,Amarāvatī,Stūpa figuré ; Buddha entourré de dévots et pi...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; panneau,bouddhisme,The British Museum,Maronet Léa
542,ASOK_AMA072_lion_14212.png,lion,"[[87, 766], [88, 805], [109, 808], [130, 808],...",0.99,ASOK_AMA072.jpg,ASOK_AMA072,Andhra Pradesh,Āndhradeśa,Amarāvatī,Stūpa figuré ; Buddha entourré de dévots et pi...,1 A.D. ; 2 A.D. ; 3 A.D.,Stūpa ; panneau,bouddhisme,The British Museum,Maronet Léa
